In [1]:
import pandas as pd

In [2]:
# Load the Dataset

df = pd.read_csv("spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
# Select Required Columns

df = df[['v1', 'v2']]
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# Rename Columns

df.columns = ['label', 'message']
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
# Understand the Dataset

df.shape

(5572, 2)

In [6]:
df.columns


Index(['label', 'message'], dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [8]:
df.isnull().sum()

label      0
message    0
dtype: int64

In [9]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [12]:
# Separate Input and Output

x = df["message"]
y = df["label"]
y.head()

0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: object

In [13]:
# Label Encoding

from sklearn.preprocessing import LabelEncoder


In [15]:
encoder = LabelEncoder()
encoder

LabelEncoder()

In [16]:
y = encoder.fit_transform(y)
y[:11]

array([0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0])

In [17]:
print(encoder.classes_)

['ham' 'spam']


In [18]:
for i, label in enumerate(encoder.classes_):
    print(label, ":", i)

ham : 0
spam : 1


In [19]:
# Train-Test Split

from sklearn.model_selection import train_test_split

In [20]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
print(x_train.shape)
print(x_test.shape)

print(y_train.shape)
print(y_test.shape)

(4457,)
(1115,)
(4457,)
(1115,)


In [22]:
# Text Preprocessing

import re

In [23]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

In [24]:
x_train = x_train.apply(clean_text)
x_test = x_test.apply(clean_text)

In [25]:
x_train.iloc[:5]

184                             going on nothing greatbye
2171                          i wont so wats wit the guys
5422                     ok ksry i knw  sivatats y i askd
4113    where are you  what do you do  how can you sta...
4588          have you not finished work yet or something
Name: message, dtype: object

In [26]:
# Feature Extraction

from sklearn.feature_extraction.text import CountVectorizer

In [27]:
vectorizer = CountVectorizer()
vectorizer

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [28]:
x_train_bow = vectorizer.fit_transform(x_train)
x_test_bow = vectorizer.transform(x_test)

In [29]:
print(x_train_bow.shape)
print(x_test_bow.shape)

(4457, 7535)
(1115, 7535)


In [30]:
# Check vocabulary

print(vectorizer.get_feature_names_out())

['aa' 'aah' 'aaniye' ... 'zoe' 'zogtorius' 'zyada']


In [31]:
print(x_train_bow[:5].toarray())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [32]:
# Model Training

from sklearn.naive_bayes import MultinomialNB

In [33]:
model = MultinomialNB()
model

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [34]:
model.fit(x_train_bow, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [35]:
y_pred = model.predict(x_test_bow)
print(y_pred[:11])

[0 0 0 1 0 0 0 0 0 0 0]


In [36]:
result = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

print(result[:21])

    Actual  Predicted
0        0          0
1        0          0
2        0          0
3        1          1
4        0          0
5        0          0
6        0          0
7        0          0
8        0          0
9        0          0
10       0          0
11       0          0
12       0          0
13       0          0
14       0          0
15       0          0
16       0          0
17       0          0
18       0          0
19       0          0
20       0          0


In [37]:
# Model Evaluation

from sklearn.metrics import accuracy_score

In [38]:
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.979372197309417


In [39]:
print("Accuracy:", accuracy * 100, "%")

Accuracy: 97.9372197309417 %


In [40]:
# Test with New Messages

new_message = ["Congratulations! You won a free lottery prize"]
new_message = [clean_text(msg) for msg in new_message]
new_message

['congratulations you won a free lottery prize']

In [41]:
new_message_vector = vectorizer.transform(new_message)
new_message_vector.toarray()

array([[0, 0, 0, ..., 0, 0, 0]], shape=(1, 7535))

In [42]:
prediction = model.predict(new_message_vector)
prediction

array([1])

In [43]:
if prediction[0] == 1:
    print('Spam')
else:
    print('Ham')

Spam


In this project, I developed an SMS Spam Detection model using Natural Language Processing and Machine Learning. I explored the dataset, performed text preprocessing, and converted messages into numerical features using the Bag of Words technique. I trained a Multinomial Naive Bayes model and evaluated its performance using accuracy. Finally, I tested the model with new SMS messages to classify them as Spam or Ham.